# Simulation studies for 'Calibration without labels in multiple testing'
Adway Wadekar

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / "src"

sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
from npeb import Grenander, local_fdr
import scipy.stats as stats
from scipy.io import loadmat
from scipy.integrate import quad
import matplotlib.pyplot as plt
from scipy.optimize import isotonic_regression
from tqdm import tqdm

from calibration_helpers.mixtures import make_pvals, conditional_mean
from calibration_helpers.misc import qvalues
from calibration_helpers.calibrators import (
  GrenanderLfdrCalibrator,
  SplineMLELfdrCalibrator,
  LindseyLfdrCalibrator,
  PValueCalibrator,
  QValueCalibrator
)
from calibration_helpers.metrics import (
    calibrate_and_assess,
)

## Example experiment (lfdr calibration)

In [ ]:
labels, pvals = make_pvals(n=50_000, pi0=0.90, alpha=0.5, beta=2.3)
qvals = qvalues(pvals, lam=0.5)

train_idx = np.random.choice(len(pvals), size=int(0.5 * len(pvals)), replace=False)
train_pvals = pvals[train_idx]
train_labels = labels[train_idx]
test_pvals = pvals[~np.isin(pvals, train_pvals)]
test_labels = labels[~np.isin(pvals, train_pvals)]

lam = 1 - len(test_pvals)**(-1/5)
pi_hat_0_gren = np.mean(test_pvals > lam) / (1 - lam)

In [ ]:
cal_gren = GrenanderLfdrCalibrator()
cal_gren.fit(train_pvals)
test_lfdr = cal_gren.predict(test_pvals)

In [ ]:
test_df = pd.DataFrame({
    'p_value': test_pvals,
    'lfdr': test_lfdr,
    'label': test_labels
})

# lfdr bins
bins = np.linspace(0, 1, 10)

# put lfdr  into bins
# df["lfdr_bin"] = pd.cut(df["lfdr"], bins=bins, include_lowest=True, right=True)
test_df["lfdr_bin"] = pd.cut(test_df["lfdr"], bins=bins, include_lowest=True, right=True)

# m = len(df)
m = len(test_df)

# summarize p-values within each lfdr-bin
lfdr_bin_stats = (
    test_df.groupby("lfdr_bin", observed=False)
      .agg(
          p_min=("p_value", "min"),
          p_max=("p_value", "max"),
          p_mean=("p_value", "mean"),
          lfdr_mean=("lfdr", "mean"),
          count=("p_value", "size"),
          proportion_true_nulls = ("label", "mean")
      )
      .reset_index()
)

lfdr_bin_stats["proportion_true_nulls"] = 1 - lfdr_bin_stats["proportion_true_nulls"]

# empirical p-range length induced by each lfdr-bin
lfdr_bin_stats["p_bin_length"] = lfdr_bin_stats["p_max"] - lfdr_bin_stats["p_min"]

# expected number of nulls in that p-interval under Storey's estimate
lfdr_bin_stats["true_nulls_hat"] = pi_hat_0_gren * m * lfdr_bin_stats["p_bin_length"]

# estimated null proportion within the lfdr-bin
lfdr_bin_stats["proportion_true_nulls_hat"] = np.minimum(
    lfdr_bin_stats["true_nulls_hat"] / lfdr_bin_stats["count"],
    1.0
)

lfdr_bin_stats["lfdr_bin_start"] = lfdr_bin_stats["lfdr_bin"].apply(lambda x: x.left)
lfdr_bin_stats["lfdr_bin_end"] = lfdr_bin_stats["lfdr_bin"].apply(lambda x: x.right)

print(lfdr_bin_stats)

# plot reliability plot for lfdr bins (estimated null proportion vs. mean lfdr in bin)

plt.figure(figsize=(8, 8))

plt.plot(lfdr_bin_stats["lfdr_mean"], lfdr_bin_stats["proportion_true_nulls_hat"], "o", label="Estimated null proportion")
plt.plot(lfdr_bin_stats["lfdr_mean"], lfdr_bin_stats["proportion_true_nulls"], "x", label="Actual null proportion")

plt.plot([0, 1], [0, 1], "k--", label="Ideal calibration")

plt.xlabel("Mean lfdr in lfdr-bin")
plt.ylabel("Estimated proportion of true nulls")
plt.title("Reliability Plot by lfdr-bin for Sucz and Ioannidis (2017) Data")

plt.legend()
plt.show()

## Experiments

In [ ]:
# run experiments for different calibrators and compare calibration errors
M = 10
pi0s = [0.75]
alphas = [0.5]
beta = 2.3
results = []
for pi0 in pi0s:
    for alpha in alphas:
        for i in tqdm(range(M)):
            for n in [1000, 10_000, 100_000, 1_000_000]:
                labels, pvals = make_pvals(
                    n=n,
                    pi0=pi0,
                    alpha=alpha,
                    beta=beta,
                )
                for calibrator in ["g-lfdr", "s-mle-lfdr", "l-lfdr", "q-value", "p-value"]:
                    calibration_error = calibrate_and_assess(
                        pvals,
                        labels,
                        calibrator=calibrator,
                        pi0=pi0,
                        alpha=alpha,
                        beta=beta,
                        use_MC=True,
                        B=10_000_000,
                    )
                    results.append({
                        "pi0": pi0,
                        "alpha": alpha,
                        "beta": beta,
                        "rep": i,
                        "n": n,
                        "calibrator": calibrator,
                        "calibration_error": calibration_error,
                    })

results_df = pd.DataFrame(results)

In [ ]:
# aggregate results by calibrator, oracle vs. non-oracle, n
results_df = pd.DataFrame(results)
summary_df = results_df.groupby(["calibrator", "n"]).agg(
    mean_brier_regret = ("calibration_error", "mean"),
    std_brier_regret = ("calibration_error", "std")
).reset_index()

summary_df

In [ ]:
# get rates of decay of calibration error with n for each calibrator by fitting a line to the log-log plot
from scipy.stats import linregress

for calibrator in summary_df["calibrator"].unique():
    subset = summary_df[summary_df["calibrator"] == calibrator]
    log_n = np.log(subset["n"])
    log_error = np.log(subset["mean_brier_regret"])
    slope, intercept, r_value, p_value, std_err = linregress(log_n, log_error)
    print(f"Calibrator: {calibrator}, Rate of decay: {slope:.2f}, R-squared: {r_value**2:.4f}")

## Analyze cluster simulations

In [ ]:
# load csv
results_df = pd.read_csv("../data/sim_summary.csv")
results_df

In [ ]:
def monotone_adjustment(pi0, alpha, beta, B):
  if alpha <= 1:
    raise ValueError("Alpha must be greater than 1 to need monotone adjustment.")
  n0 = int(pi0 * B)
  n1 = B - n0
  p_vals = np.concatenate([np.random.uniform(size=n0), np.random.beta(a=alpha, b=beta, size=n1)])
  p_vals = np.sort(p_vals)

  lfdr_mc = conditional_mean(p_vals, pi0, alpha, beta)
  lfdr_iso = isotonic_regression(lfdr_mc, increasing=True)
  lfdr_uparrow_mc = lfdr_iso.x

  return np.mean((lfdr_uparrow_mc - lfdr_mc)**2)

In [ ]:
# for alpha = 1.5 in results_df, subtract off monotone adjustment error from mean brier regret, for each beta and pi_0
B = 10_000_000

# subtract off 
results_df["monotone_adjustment_error"] = results_df.apply(lambda row: monotone_adjustment(row["pi0"], row["alpha"], row["beta"], B) if row["alpha"] > 1 else 0, axis=1)
results_df["mean_brier_regret_adj"] = results_df["mean_calibration_error"] - results_df["monotone_adjustment_error"]

In [ ]:
# save results_df to pickle
results_df.to_pickle("../data/sim_summary.pkl")

If starting from the pkl file, just load the pkl file here and start at this point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import LogLocator, NullFormatter

plot_df = results_df[results_df["alpha"] != 0.7].copy()

pis = np.sort(plot_df["pi0"].unique())
alphas = np.sort(plot_df["alpha"].unique())

calibrators = ['q-value', 'p-value', 'lfdr']

xmin = 5e2
xmax = 1e5

# Modern ML-paper palette: distinct, clean, colorblind-conscious
palette = [
    "#3B82F6",  # blue
    "#EF4444",  # red
    "#10B981",  # emerald
    "#8B5CF6",  # violet
    "#F59E0B",  # amber
    "#06B6D4",  # cyan
    "#EC4899",  # pink
    "#111827",  # near-black
]

linestyles = ["-", "-", "-", "-", "--", "--", "--", "--"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*"]

color_map = dict(zip(calibrators, palette[:len(calibrators)]))
style_map = dict(zip(calibrators, linestyles[:len(calibrators)]))
marker_map = dict(zip(calibrators, markers[:len(calibrators)]))

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 12,
    "axes.titlesize": 9,
    "axes.labelsize": 10,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, axes = plt.subplots(
    nrows=len(pis),
    ncols=len(alphas),
    figsize=(8, 6),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for i, pi0 in enumerate(pis):
    for j, alpha in enumerate(alphas):
        ax = axes[i, j]

        subset = plot_df[
            (plot_df["pi0"] == pi0) &
            (plot_df["alpha"] == alpha)
        ]

        for calibrator in calibrators:
            calib_subset = subset[
                subset["calibrator"] == calibrator
            ].sort_values("n")

            if calib_subset.empty:
                continue

            x = calib_subset["n"].to_numpy()
            y = calib_subset["mean_brier_regret_adj"].to_numpy()
            se = calib_subset["sd_calibration_error"].to_numpy() / np.sqrt(500)

            ax.plot(
                x,
                y,
                color=color_map[calibrator],
                linestyle=style_map[calibrator],
                marker=marker_map[calibrator],
                markersize=3.2,
                markeredgewidth=0,
                linewidth=1.9,
                alpha=0.96,
                label=calibrator,
            )

            ax.fill_between(
                x,
                np.maximum(y - se, 1e-16),
                y + se,
                color=color_map[calibrator],
                alpha=0.10,
                linewidth=0,
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(xmin, xmax)

        ax.xaxis.set_major_locator(LogLocator(base=10))
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_minor_formatter(NullFormatter())

        ax.set_title(rf"$\pi_0={pi0},\ \alpha={alpha}$", pad=5)

        ax.grid(
            True,
            which="major",
            axis="y",
            color="#E5E7EB",
            linewidth=0.7,
            alpha=0.9,
        )

        ax.tick_params(axis="both", which="major", length=3, width=0.7, color="#374151")
        ax.tick_params(axis="both", which="minor", length=0)

        for spine in ["left", "bottom"]:
            ax.spines[spine].set_color("#374151")
            ax.spines[spine].set_linewidth(0.7)

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=min(len(labels), 4),
    frameon=False,
    bbox_to_anchor=(0.5, 1.04),
    handlelength=2.6,
    columnspacing=1.6,
)

fig.text(
    0.5, -0.025,
    r"Number of tests (log scale)",
    ha="center",
    fontsize=10,
)

fig.text(
    -0.025, 0.5,
    "Expected Brier regret (log scale)",
    va="center",
    rotation="vertical",
    fontsize=10,
)

plt.show()

In [ ]:
fig.savefig("calibration_error_grid.pdf", bbox_inches="tight")
fig.savefig("calibration_error_grid.png", dpi=300, bbox_inches="tight")

In [ ]:
# calculate rates of decay for each calibrator, pi0, alpha by fitting a line to the log-log plot of calibration error vs. n
from scipy.stats import linregress
decay_results = []
for calibrator in plot_df["calibrator"].unique():
    for pi0 in plot_df["pi0"].unique():
        for alpha in plot_df["alpha"].unique():
            subset = plot_df[
                (plot_df["calibrator"] == calibrator) &
                (plot_df["pi0"] == pi0) &
                (plot_df["alpha"] == alpha)
            ].sort_values("n")

            if subset.empty:
                continue

            log_n = np.log(subset["n"])
            log_error = np.log(subset["mean_brier_regret_adj"])
            slope, intercept, r_value, p_value, std_err = linregress(log_n, log_error)

            decay_results.append({
                "calibrator": calibrator,
                "pi0": pi0,
                "alpha": alpha,
                "decay_rate": slope,
                "std_err": std_err,
                "r_squared": r_value**2,
            })

decay_df = pd.DataFrame(decay_results)
print(decay_df)